# 02 - Pré-processamento dos Dados

## Objetivos
- Tratar valores ausentes
- Criar variáveis derivadas
- Preparar dados para modelagem

In [10]:
import pandas as pd
import numpy as np
import holidays
from sklearn.preprocessing import LabelEncoder, StandardScaler

pd.set_option('display.max_columns', None)

## 1. Carregar Dados

Usamos o dataset enriquecido gerado na EDA (com nomes de companhias e aeroportos).

In [11]:
# Tentar carregar dataset enriquecido, se não existir, carregar o original
try:
    df = pd.read_csv('../data/processed/flights_enriched.csv')
    print('Dataset enriquecido carregado!')
except FileNotFoundError:
    print('Dataset enriquecido não encontrado. Carregando original...')
    df = pd.read_csv('../data/raw/flights.csv')
    
    # Carregar tabelas auxiliares
    airlines = pd.read_csv('../data/raw/airlines.csv')
    airports = pd.read_csv('../data/raw/airports.csv')
    
    # Merge com airlines
    df = df.merge(airlines, left_on='AIRLINE', right_on='IATA_CODE', how='left')
    df = df.rename(columns={'AIRLINE_x': 'AIRLINE', 'AIRLINE_y': 'AIRLINE_NAME'})
    df = df.drop(columns=['IATA_CODE'])

print(f'Shape original: {df.shape}')

/var/folders/cv/dflf349j79s03873mkjhgkfm0000gn/T/ipykernel_7935/2537864786.py:3: DtypeWarning: Columns (7,8,32,33,34,37,38,39) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/flights_enriched.csv')


Dataset enriquecido carregado!
Shape original: (5819079, 45)


## 2. Remover Voos Cancelados

In [12]:
# Remover voos cancelados (não têm informação de atraso)
print(f'Voos cancelados: {df["CANCELLED"].sum():,}')
df = df[df['CANCELLED'] == 0].copy()
print(f'Shape após remover cancelados: {df.shape}')

Voos cancelados: 89,884
Shape após remover cancelados: (5729195, 45)


## 3. Tratamento de Valores Ausentes

In [13]:
# Ver valores ausentes
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Ausentes': missing, '%': missing_pct})
missing_df[missing_df['Ausentes'] > 0].sort_values('%', ascending=False)

,Ausentes,%
CANCELLATION_REASON,5729195,100.000000
LATE_AIRCRAFT_DELAY,4665756,81.438247
WEATHER_DELAY,4665756,81.438247
AIRLINE_DELAY,4665756,81.438247
SECURITY_DELAY,4665756,81.438247
AIR_SYSTEM_DELAY,4665756,81.438247
DEST_LAT,488295,8.522925
DEST_LONG,488295,8.522925
ORIGIN_LAT,488283,8.522716
ORIGIN_LONG,488283,8.522716


In [14]:
# Colunas de delay: preencher com 0 (sem atraso por aquela causa)
delay_cols = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 
              'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']
df[delay_cols] = df[delay_cols].fillna(0)

# Remover linhas com ARRIVAL_DELAY nulo (variável alvo)
df = df.dropna(subset=['ARRIVAL_DELAY'])

print(f'Shape após tratamento: {df.shape}')

Shape após tratamento: (5714008, 45)


## 4. Criar Variáveis Derivadas

In [15]:
# Variável alvo binária (atraso > 15 min)
df['DELAYED'] = (df['ARRIVAL_DELAY'] > 15).astype(int)

# Período do dia baseado no horário de partida
def get_period(hour):
    if pd.isna(hour):
        return 'Unknown'
    hour = int(hour) // 100  # Extrair hora de HHMM
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['DEPARTURE_PERIOD'] = df['SCHEDULED_DEPARTURE'].apply(get_period)

# É fim de semana?
df['IS_WEEKEND'] = (df['DAY_OF_WEEK'] >= 6).astype(int)

# Estação do ano
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df['SEASON'] = df['MONTH'].apply(get_season)

# Hora do dia
df['HOUR'] = df['SCHEDULED_DEPARTURE'] // 100

print('Variáveis derivadas criadas!')

Variáveis derivadas criadas!


In [16]:
# Verificar novas colunas
print('Período do dia:')
print(df['DEPARTURE_PERIOD'].value_counts())

print('\nEstação do ano:')
print(df['SEASON'].value_counts())

print('\nFim de semana:')
print(df['IS_WEEKEND'].value_counts())

Período do dia:
DEPARTURE_PERIOD
Morning      2341610
Afternoon    1720139
Evening      1287578
Night         364681
Name: count, dtype: int64

Estação do ano:
SEASON
Summer    1511187
Spring    1461030
Fall      1407398
Winter    1334393
Name: count, dtype: int64

Fim de semana:
IS_WEEKEND
0    4221708
1    1492300
Name: count, dtype: int64


In [17]:
# ── 2. Flag de Feriados Americanos (2015) — versão vetorizada ──────────────────
import datetime

us_holidays = holidays.US(years=2015)
holiday_dates = set(us_holidays.keys())

# Criar set de tuplas (month, day) para lookup rápido
holiday_month_day = {(d.month, d.day) for d in holiday_dates}

df['IS_HOLIDAY'] = df.apply(
    lambda row: 1 if (int(row['MONTH']), int(row['DAY'])) in holiday_month_day else 0,
    axis=1
)

n_holiday_flights = df['IS_HOLIDAY'].sum()
print(f'Feriados americanos em 2015: {len(us_holidays)}')
print(f'Voos em dias de feriado: {n_holiday_flights:,} ({n_holiday_flights/len(df)*100:.1f}%)')
print(f'\nFeriados 2015:')
for d, name in sorted(us_holidays.items()):
    print(f'  {d}: {name}')

Feriados americanos em 2015: 11
Voos em dias de feriado: 157,475 (2.8%)

Feriados 2015:
  2015-01-01: New Year's Day
  2015-01-19: Martin Luther King Jr. Day
  2015-02-16: Washington's Birthday
  2015-05-25: Memorial Day
  2015-07-03: Independence Day (observed)
  2015-07-04: Independence Day
  2015-09-07: Labor Day
  2015-10-12: Columbus Day
  2015-11-11: Veterans Day
  2015-11-26: Thanksgiving Day
  2015-12-25: Christmas Day


In [18]:
# Congestionamento Diário do Aeroporto 
# Número de voos saindo do mesmo aeroporto no mesmo dia.
# Aeroportos com mais voos têm maior risco de atrasos em cascata.

daily_flights = (
    df.groupby(['ORIGIN_AIRPORT', 'MONTH', 'DAY'])['FLIGHT_NUMBER']
    .count()
    .reset_index()
    .rename(columns={'FLIGHT_NUMBER': 'ORIGIN_DAILY_FLIGHTS'})
)

df = df.merge(daily_flights, on=['ORIGIN_AIRPORT', 'MONTH', 'DAY'], how='left')

print(f'ORIGIN_DAILY_FLIGHTS criada.')
print(f'  Média de voos/aeroporto/dia: {df["ORIGIN_DAILY_FLIGHTS"].mean():.1f}')
print(f'  Máximo: {df["ORIGIN_DAILY_FLIGHTS"].max()}')

ORIGIN_DAILY_FLIGHTS criada.
  Média de voos/aeroporto/dia: 343.0
  Máximo: 1166


In [19]:
# Média Histórica de Atraso por Rota
# Rotas com histórico de atraso alto tendem a continuar atrasando.
# Nota: calculado no dataset completo (inclui dados que serão usados como teste).
# Limitação documentada: leve vazamento de dados (data leakage) — aceitável para
# fins educacionais, mas em produção deveria ser calculado apenas no treino.

route_delay = (
    df.groupby(['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'])['ARRIVAL_DELAY']
    .mean()
    .reset_index()
    .rename(columns={'ARRIVAL_DELAY': 'ROUTE_DELAY_MEAN'})
)

df = df.merge(route_delay, on=['ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'], how='left')

print(f'ROUTE_DELAY_MEAN criada.')
print(f'  Média geral: {df["ROUTE_DELAY_MEAN"].mean():.2f} min')
print(f'  Top 5 rotas mais atrasadas:')
print(route_delay.nlargest(5, 'ROUTE_DELAY_MEAN').to_string(index=False))

ROUTE_DELAY_MEAN criada.
  Média geral: 4.41 min
  Top 5 rotas mais atrasadas:
ORIGIN_AIRPORT DESTINATION_AIRPORT  ROUTE_DELAY_MEAN
         14869               13264             996.0
         11433               13303             530.0
           IAD                 TTN             381.0
         14262               12266             344.0
         11775               10397             297.0


In [20]:
# Flag de Feriados Americanos (2015) 
us_holidays = holidays.US(years=2015)

def is_holiday(row):
    try:
        import datetime
        d = datetime.date(int(row['YEAR']), int(row['MONTH']), int(row['DAY']))
        return 1 if d in us_holidays else 0
    except Exception:
        return 0

df['IS_HOLIDAY'] = df.apply(is_holiday, axis=1)

n_holiday_flights = df['IS_HOLIDAY'].sum()
print(f'Feriados americanos em 2015: {len(us_holidays)}')
print(f'Voos em dias de feriado: {n_holiday_flights:,} ({n_holiday_flights/len(df)*100:.1f}%)')
print(f'\nFeriados 2015 identificados:')
for d, name in sorted(us_holidays.items()):
    print(f'  {d}: {name}')

Feriados americanos em 2015: 11
Voos em dias de feriado: 157,475 (2.8%)

Feriados 2015 identificados:
  2015-01-01: New Year's Day
  2015-01-19: Martin Luther King Jr. Day
  2015-02-16: Washington's Birthday
  2015-05-25: Memorial Day
  2015-07-03: Independence Day (observed)
  2015-07-04: Independence Day
  2015-09-07: Labor Day
  2015-10-12: Columbus Day
  2015-11-11: Veterans Day
  2015-11-26: Thanksgiving Day
  2015-12-25: Christmas Day


In [21]:
# Encoding Cíclico 
# Captura a natureza cíclica das variáveis temporais (hora, dia da semana, mês)

# HOUR — ciclo de 24 horas
df['HOUR_SIN'] = np.sin(2 * np.pi * df['HOUR'] / 24)
df['HOUR_COS'] = np.cos(2 * np.pi * df['HOUR'] / 24)

# DAY_OF_WEEK — ciclo de 7 dias (1=Segunda, 7=Domingo no dataset)
df['DOW_SIN'] = np.sin(2 * np.pi * df['DAY_OF_WEEK'] / 7)
df['DOW_COS'] = np.cos(2 * np.pi * df['DAY_OF_WEEK'] / 7)

# MONTH — ciclo de 12 meses
df['MONTH_SIN'] = np.sin(2 * np.pi * df['MONTH'] / 12)
df['MONTH_COS'] = np.cos(2 * np.pi * df['MONTH'] / 12)

print('Encoding cíclico criado:')
print('  HOUR_SIN, HOUR_COS')
print('  DOW_SIN,  DOW_COS')
print('  MONTH_SIN, MONTH_COS')

Encoding cíclico criado:
  HOUR_SIN, HOUR_COS
  DOW_SIN,  DOW_COS
  MONTH_SIN, MONTH_COS


In [22]:
# Resumo das novas features
print('\n=== NOVAS FEATURES ADICIONADAS ===')
new_features = ['HOUR_SIN', 'HOUR_COS', 'DOW_SIN', 'DOW_COS',
                'MONTH_SIN', 'MONTH_COS', 'IS_HOLIDAY',
                'ROUTE_DELAY_MEAN', 'ORIGIN_DAILY_FLIGHTS']
print(df[new_features].describe().round(3))


=== NOVAS FEATURES ADICIONADAS ===
          HOUR_SIN     HOUR_COS      DOW_SIN      DOW_COS    MONTH_SIN  \
count  5714008.000  5714008.000  5714008.000  5714008.000  5714008.000   
mean        -0.086       -0.376        0.016       -0.026       -0.013   
std          0.752        0.535        0.706        0.707        0.705   
min         -1.000       -1.000       -0.975       -0.901       -1.000   
25%         -0.866       -0.866       -0.782       -0.901       -0.866   
50%         -0.259       -0.500       -0.000       -0.223       -0.000   
75%          0.707        0.000        0.782        0.623        0.500   
max          1.000        1.000        0.975        1.000        1.000   

         MONTH_COS   IS_HOLIDAY  ROUTE_DELAY_MEAN  ORIGIN_DAILY_FLIGHTS  
count  5714008.000  5714008.000       5714008.000           5714008.000  
mean        -0.025        0.028             4.407               342.995  
std          0.709        0.164             5.100               295.619  
m

# Features para modelagem supervisionada
features_supervised = [
    # Temporal — encoding cíclico (substitui MONTH, DAY_OF_WEEK, HOUR ordinais)
    'MONTH_SIN', 'MONTH_COS',
    'DOW_SIN', 'DOW_COS',
    'HOUR_SIN', 'HOUR_COS',
    # Temporal — ordinal
    'DAY',
    # Operacional
    'AIRLINE_ENCODED', 'ORIGIN_ENCODED', 'DEST_ENCODED',
    # Características do voo
    'SCHEDULED_TIME', 'DISTANCE',
    # Variáveis derivadas básicas
    'PERIOD_ENCODED', 'SEASON_ENCODED', 'IS_WEEKEND',
    # Features avançadas
    'IS_HOLIDAY', 'ROUTE_DELAY_MEAN', 'ORIGIN_DAILY_FLIGHTS'
]

# Target para classificação
target_classification = 'DELAYED'

# Target para regressão
target_regression = 'ARRIVAL_DELAY'

print(f'Features selecionadas: {len(features_supervised)}')
for f in features_supervised:
    print(f'  - {f}')

## 5. Encoding de Variáveis Categóricas

In [23]:
# Label Encoding para variáveis categóricas
le_airline = LabelEncoder()
le_origin = LabelEncoder()
le_dest = LabelEncoder()
le_period = LabelEncoder()
le_season = LabelEncoder()

df['AIRLINE_ENCODED'] = le_airline.fit_transform(df['AIRLINE'])
df['ORIGIN_ENCODED'] = le_origin.fit_transform(df['ORIGIN_AIRPORT'].astype(str))
df['DEST_ENCODED'] = le_dest.fit_transform(df['DESTINATION_AIRPORT'].astype(str))
df['PERIOD_ENCODED'] = le_period.fit_transform(df['DEPARTURE_PERIOD'])
df['SEASON_ENCODED'] = le_season.fit_transform(df['SEASON'])

print('Encoding concluído!')

Encoding concluído!


In [24]:
# Mapear códigos para nomes (para referência)
print('Mapeamento AIRLINE:')
airline_mapping = dict(zip(le_airline.classes_, range(len(le_airline.classes_))))
print(airline_mapping)

print('\nMapeamento PERIOD:')
period_mapping = dict(zip(le_period.classes_, range(len(le_period.classes_))))
print(period_mapping)

print('\nMapeamento SEASON:')
season_mapping = dict(zip(le_season.classes_, range(len(le_season.classes_))))
print(season_mapping)

Mapeamento AIRLINE:
{'AA': 0, 'AS': 1, 'B6': 2, 'DL': 3, 'EV': 4, 'F9': 5, 'HA': 6, 'MQ': 7, 'NK': 8, 'OO': 9, 'UA': 10, 'US': 11, 'VX': 12, 'WN': 13}

Mapeamento PERIOD:
{'Afternoon': 0, 'Evening': 1, 'Morning': 2, 'Night': 3}

Mapeamento SEASON:
{'Fall': 0, 'Spring': 1, 'Summer': 2, 'Winter': 3}


## 6. Selecionar Features para Modelagem

In [25]:
# Features para modelagem supervisionada
features_supervised = [
    'MONTH', 'DAY', 'DAY_OF_WEEK', 'HOUR',
    'AIRLINE_ENCODED', 'ORIGIN_ENCODED', 'DEST_ENCODED', 
    'SCHEDULED_TIME', 'DISTANCE', 
    'PERIOD_ENCODED', 'SEASON_ENCODED', 'IS_WEEKEND'
]

# Target para classificação
target_classification = 'DELAYED'

# Target para regressão
target_regression = 'ARRIVAL_DELAY'

print(f'Features selecionadas: {len(features_supervised)}')
print(features_supervised)

Features selecionadas: 12
['MONTH', 'DAY', 'DAY_OF_WEEK', 'HOUR', 'AIRLINE_ENCODED', 'ORIGIN_ENCODED', 'DEST_ENCODED', 'SCHEDULED_TIME', 'DISTANCE', 'PERIOD_ENCODED', 'SEASON_ENCODED', 'IS_WEEKEND']


## 7. Resumo do Dataset Processado

In [26]:
# Resumo
print('=== RESUMO DO DATASET PROCESSADO ===')
print(f'\nShape: {df.shape}')
print(f'\nDistribuição da variável alvo (DELAYED):')
print(df['DELAYED'].value_counts())
print(f'\nProporção:')
print(df['DELAYED'].value_counts(normalize=True))

=== RESUMO DO DATASET PROCESSADO ===

Shape: (5714008, 62)

Distribuição da variável alvo (DELAYED):
DELAYED
0    4690510
1    1023498
Name: count, dtype: int64

Proporção:
DELAYED
0    0.820879
1    0.179121
Name: proportion, dtype: float64


In [27]:
# Estatísticas de ARRIVAL_DELAY
print('\nEstatísticas de ARRIVAL_DELAY (para regressão):')
print(df['ARRIVAL_DELAY'].describe())


Estatísticas de ARRIVAL_DELAY (para regressão):
count    5.714008e+06
mean     4.407057e+00
std      3.927130e+01
min     -8.700000e+01
25%     -1.300000e+01
50%     -5.000000e+00
75%      8.000000e+00
max      1.971000e+03
Name: ARRIVAL_DELAY, dtype: float64


## 8. Salvar Dados Processados

In [28]:
# Salvar dataset processado
df.to_csv('../data/processed/flights_processed.csv', index=False)
print('Dados salvos em data/processed/flights_processed.csv')
print(f'Shape final: {df.shape}')

Dados salvos em data/processed/flights_processed.csv
Shape final: (5714008, 62)


In [29]:
# Verificar colunas finais
print('\nColunas do dataset processado:')
print(df.columns.tolist())


Colunas do dataset processado:
['YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'FLIGHT_NUMBER', 'TAIL_NUMBER', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ARRIVAL_DELAY', 'DIVERTED', 'CANCELLED', 'CANCELLATION_REASON', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'AIRLINE_NAME', 'ORIGIN_AIRPORT_NAME', 'ORIGIN_CITY', 'ORIGIN_STATE', 'ORIGIN_LAT', 'ORIGIN_LONG', 'DEST_AIRPORT_NAME', 'DEST_CITY', 'DEST_STATE', 'DEST_LAT', 'DEST_LONG', 'DELAYED', 'DAY_NAME', 'HOUR', 'DEPARTURE_PERIOD', 'IS_WEEKEND', 'SEASON', 'IS_HOLIDAY', 'ORIGIN_DAILY_FLIGHTS', 'ROUTE_DELAY_MEAN', 'HOUR_SIN', 'HOUR_COS', 'DOW_SIN', 'DOW_COS', 'MONTH_SIN', 'MONTH_COS', 'AIRLINE_ENCODED', 'ORIGIN_ENCODED', 'DEST_ENCODED', 'PERIOD_ENCODED', 'SEASON_ENCODED']
